<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
LibreChat
</font>
</h1>

In [ ]:
from langchain_community.document_loaders import PyPDFium2Loader

loader = PyPDFium2Loader("data/justforfun_persian.pdf")
pages = loader.load()

In [ ]:
print("Number of pages (loaded Document objects):", len(pages))

In [ ]:
print(pages[15].page_content)

In [ ]:
import re

def clean_persian_pdf_text(text):
    if not text:
        return ""
    
    replacements = {
        'ͬ': 'ی',  # اصلاح مͬ -> می، ریاضͬ -> ریاضی
        'ͷ': 'ک',  # اصلاح یͷ -> یک، کمͷ -> کمک
        'ͺ': 'ک',  # اصلاح امͺان -> امکان
        'ͽ': 'گ',  # اصلاح دیͽری -> دیگری
    }
    
    for bad_char, good_char in replacements.items():
        text = text.replace(bad_char, good_char)
        
    # تمیزکاری فواصل اضافی
    text = re.sub(r' +', ' ', text)
    return text.strip()

# اعمال اصلاحات روی تمام ۲۰۴ صفحه بارگذاری شده
for doc in pages:
    doc.page_content = clean_persian_pdf_text(doc.page_content)


print("متن اصلاح شده صفحه ۱۵:")
print(pages[15].page_content[:300])

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

target_url = "https://linuxbook.ir/all.html"
loader_simple = WebBaseLoader(target_url)
web_docs = loader_simple.load()

print(web_docs[0].page_content[:512])

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

target_url = "https://linuxbook.ir/all.html"
loader_clean = WebBaseLoader(
    target_url ,
    bs_kwargs = {
        "parse_only" : bs4.SoupStrainer("main")
    }
)
web_docs_more_clean = loader_clean.load()

In [ ]:
if 'web_docs_more_clean' in locals() and web_docs_more_clean:
    print("تعداد صفحات وب پالایش شده:", len(web_docs_more_clean))
    print("نمونه متن وب (۱۰۰ کاراکتر اول):")
    print(web_docs_more_clean[0].page_content[:100])
else:
    print("متغیر web_docs_more_clean هنوز به درستی تعریف یا اجرا نشده است.")

In [ ]:
import requests
import time
import random
from langchain_core.documents import Document

wiki_titles = ['ریچارد استالمن', 'لینوس توروالدز', 'لینوکس', 'پروژه گنو', 'نرم‌افزار آزاد', 'بنیاد نرم‌افزار آزاد']
wiki_docs = []

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

for title in wiki_titles:
    print(f"در حال بارگیری مستقیم از API ویکی‌پدیا: {title}...")
    url = "https://fa.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "titles": title,
        "redirects": 1
    }
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            pages = data.get("query", {}).get("pages", {})
            
            for page_id, page_info in pages.items():
                if page_id != "-1":
                    page_title = page_info.get("title")
                    page_content = page_info.get("extract", "")
                    
                    doc = Document(
                        page_content=page_content,
                        metadata={"source": f"https://fa.wikipedia.org/wiki/{page_title}", "title": page_title}
                    )
                    wiki_docs.append(doc)
                    print(f" '{page_title}' با موفقیت دریافت و به داکیومنت تبدیل شد.")
                else:
                    print(f" صفحه‌ای با عنوان دقیق '{title}' در ویکی‌پدیای فارسی پیدا نشد.")
        else:
            print(f" سرور ویکی‌پدیا پاسخ نداد. کد خطا: {response.status_code}")
            

        time.sleep(random.uniform(1.5, 3))
        
    except Exception as e:
        print(f"خطا در شبکه یا ساختار داده: {e}")
        time.sleep(3)

print(f"\nعملیات تمام شد! تعداد کل داکیومنت‌های ذخیره شده در wiki_docs: {len(wiki_docs)}")

In [ ]:
print("Number of wikipedia pages (loaded Document objects):", len(wiki_docs))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import BSHTMLLoader


loader = DirectoryLoader(
    path="data\html",
    glob="*.html",
    loader_cls=BSHTMLLoader
)

html_docs = loader.load()

print(f"تعداد فایل‌های HTML بارگیری شده با موفقیت: {len(html_docs)}")

In [ ]:
print("Number of pages (loaded Document objects):", len(html_docs))

In [ ]:
all_documents = []


if 'web_docs_more_clean' in locals() and web_docs_more_clean:
    all_documents.extend(web_docs_more_clean)
    print(f"تعداد {len(web_docs_more_clean)} سند از وب اضافه شد.")

if 'wiki_docs' in locals() and wiki_docs:
    all_documents.extend(wiki_docs)
    print(f"تعداد {len(wiki_docs)} سند از ویکی‌پدیا اضافه شد.")

if 'html_docs' in locals() and html_docs:
    all_documents.extend(html_docs)
    print(f"تعداد {len(html_docs)} سند از HTML استالمن اضافه شد.")

if 'pages' in locals() and pages:
    all_documents.extend(pages)
    print(f"تعداد {len(pages)} صفحه از کتاب PDF اضافه شد.")


print(f"\nمجموع کل اسناد تجمیع شده در all_documents: {len(all_documents)}")

In [ ]:
all_documents

In [ ]:
import re

STALLMAN_FOOTER_PATTERNS = [
    r"Please send comments on these web pages to rms@gnu\.org\.?",
    r"Copyright \(C\)\s*\d{4}.*?Richard Stallman",
    r"Copyright \(c\)\s*\d{4}.*?Richard Stallman",
    r"Verbatim copying and (?:redistribution|distrib\w*) of this entire page (?:is|are) permitted.*?(?:\.|$)",
    r"Return to Richard Stallman'?s home page\.?",
]

LINUXBOOK_NAV_PATTERNS = [
    r"Toggle navigation",
    r"^\s*روی جلد\s*$",
    r"^\s*درباره\s*$",
    r"^\s*دانلود\s*$",
    r"^\s*حمایت\s*$",
]

def clean_document_text(text: str) -> str:
    """حذف فوترها/ناوبری‌های تکراری از متن یک سند، قبل از چانک‌بندی."""
    cleaned = text

    for pattern in STALLMAN_FOOTER_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE | re.DOTALL)

    for pattern in LINUXBOOK_NAV_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE | re.MULTILINE)

    # جمع کردن فاصله‌ها و خط‌های خالی اضافه‌ای که از حذف بالا باقی می‌ماند
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\n\s*\n+", "\n\n", cleaned)
    return cleaned.strip()


def deduplicate_documents(docs):
    """حذف اسنادی که محتوای آن‌ها (پس از نرمال‌سازی فاصله) عیناً تکراری است.

    این کار در سطح «سند کامل» انجام می‌شود، نه چانک -- یعنی قبل از چانک‌بندی.
    اگر دو Document متفاوت (حتی با منبع متفاوت) دقیقاً همان متن را داشته باشند
    فقط نسخه‌ی اول نگه داشته می‌شود.
    """
    seen = set()
    unique_docs = []
    removed = 0
    for doc in docs:
        normalized = " ".join(doc.page_content.split())
        if not normalized:
            continue
        if normalized in seen:
            removed += 1
            continue
        seen.add(normalized)
        unique_docs.append(doc)
    print(f"حذف {removed} سند کاملاً تکراری از مجموع {len(docs)} سند (باقی‌مانده: {len(unique_docs)})")
    return unique_docs


def clean_documents(docs):
    """اعمال clean_document_text روی page_content همه‌ی اسناد یک لیست."""
    cleaned_docs = []
    for doc in docs:
        doc.page_content = clean_document_text(doc.page_content)
        if doc.page_content:
            cleaned_docs.append(doc)
    return cleaned_docs


print(" توابع پاک‌سازی و حذف تکرار اسناد تعریف شدند.")

cleaned_all_documents = clean_documents(all_documents)

final_documents = deduplicate_documents(cleaned_all_documents)

print(f"فرآیند پالایش به پایان رسید. تعداد {len(final_documents)} سند نهایی و یکتا آماده خرد شدن هستند.")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

# شکستن اسناد به چانک‌های کوچک‌تر
chunks = text_splitter.split_documents(final_documents)

for chunk in chunks:
    chunk.page_content = f"passage: {chunk.page_content}"

print(f"تعداد چانک‌های ایجاد شده: {len(chunks)}")
print("نمونه چانک اول با پیشوند:")
print(chunks[0].page_content[:200] + "...")

In [ ]:
import os
import shutil
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

local_model_path = "PATH OF YOUR MODEL"

encode_kwargs = {'normalize_embeddings': True}

print(f"Loading embedding model locally from: {local_model_path}")
embedding_model = HuggingFaceEmbeddings(
    model_name=local_model_path,
    encode_kwargs=encode_kwargs
)

persist_directory = "./chroma_db"
if os.path.exists(persist_directory):
    print("Found old database. Clearing directory for a clean slate...")
    shutil.rmtree(persist_directory)
    time.sleep(1)

print("Building Chroma Vector Store from local model...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print("Vector database successfully built and persisted locally!")

In [ ]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    base_url='BASE URL',
    api_key='YOUR TOKEN',
    model="YOUR MODEL",
    temperature=0.2
)

print("با موفقیت به مدل زبانی متصل شد")

In [ ]:
# تست سلامت و اصالت اتصال به دیپ‌سیک
try:
    response = llm.invoke("سلام خوبی؟")
    print(f"عالیه! ارتباط مستقیم برقرار شد. پاسخ مدل:\n{response.content}")
except Exception as e:
    print(f"خطایی در اتصال رخ داده است. احتمالاً کلید API را ناقص کپی کردی یا اعتبارت تمام شده:\n{e}")

In [ ]:
import json
import re
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser



class RagResponse(BaseModel):
    question_number: int = Field(description="The exact number of the question")
    answer: str = Field(description="The precise answer from context or 'در متن یافت نشد'")

parser = JsonOutputParser(pydantic_object=RagResponse)


def clean_and_get_number(raw_query):
    """استخراج شماره سوال و پاک‌سازی متن اصلی کوئری"""
    q_num_match = re.search(r'پرسش\s*([0-9\d۰-۹]+)', raw_query)
    if q_num_match:
        raw_num = q_num_match.group(1)
        en_num = "".join(str("0123456789۰۱۲۳۴۵۶۷۸۹".index(d)%10) for d in raw_num)
        q_num = int(en_num)
    else:
        q_num = 1
    cleaned_q = re.sub(r'^پرسش\s*[0-9\d۰-۹]+\s*[:：]\s*', '', raw_query).strip()
    return q_num, cleaned_q

def base_retrieval(cleaned_query):
    """گام اول: بازیابی اولیه (تعداد بالا برای باز بودن دست رنکر)"""

    e5_query = f"query: {cleaned_query}"
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 10})
    return retriever.invoke(e5_query)

def llm_reranker(docs, cleaned_query):
    """گام دوم: رتبه‌بندی مجدد و انتخاب دقیق‌ترین چانک‌ها"""
    if not docs:
        return []
        
    rerank_prompt = """You are an expert reranker. Given the query and a list of text chunks, select the top 4 most relevant chunks that directly help answer the query.
Return your response as a JSON object containing a list of the 0-based indices of the top chunks, ordered by relevance.
Example Format: {{"top_indices": [2, 0, 5, 1]}}

Query: {query}

Chunks:
{chunks_str}"""

    chunks_str = "\n".join(f"[{i}]: {doc.page_content[:350]}\n---" for i, doc in enumerate(docs))
    
    try:
        response = llm.invoke(rerank_prompt.format(query=cleaned_query, chunks_str=chunks_str))
        res_text = response.content if isinstance(response.content, str) else response.content.get("text", "")
        
        match = re.search(r'\{.*\}', res_text, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            top_indices = data.get("top_indices", [])

            return [docs[i] for i in top_indices if i < len(docs)][:4]
    except Exception:
        pass
    
    return docs[:4]


def process_rag_flow(inputs):
    raw_query = inputs["query"]
    q_num, cleaned_q = clean_and_get_number(raw_query)
    
    raw_docs = base_retrieval(cleaned_q)

    reranked_docs = llm_reranker(raw_docs, cleaned_q)
    
    context_str = "\n\n".join(doc.page_content for doc in reranked_docs) if reranked_docs else "(متنی یافت نشد)"
    
    return {
        "context": context_str,
        "query": cleaned_q,
        "question_number": q_num,
        "format_instructions": parser.get_format_instructions()
    }


system_prompt = """You are an expert QA system for a GNU/Linux RAG pipeline. Your task is to answer the user's question BASED ONLY on the provided Context.

CRITICAL RULES:
1. If the question asks for an acronym/abbreviation (e.g., FHS, POSIX, GNU), you MUST output that acronym in uppercase English letters. Do not transliterate it to Persian.
2. If the exact answer is not in the text, strictly return 'در متن یافت نشد'.
3. Do not assume or extrapolate. Clear, exact, and concise.

\n{format_instructions}\n
Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Question Number: {question_number}\nQuestion: {query}")
])


rag_chain = (
    RunnablePassthrough()
    | process_rag_flow
    | prompt
    | llm
    | parser
)

print("پایپ‌لاین کلاسیک Base Retrieval ➔ Reranking با موفقیت مستقر شد!")

In [ ]:
query = "پرسش ۱: توروالدز برای کار در چه موسسه‌ای دانشگاه هلسینکی را ترک گفت؟"
answer1 = rag_chain.invoke({"query": query})
answer1

In [ ]:
query = "پرسش ۲: آندرو تاننباوم استاد کدام دانشگاه است؟"
answer2 = rag_chain.invoke({"query": query})
answer2

In [ ]:
query = "پرسش ۳: در سال ۲۰۰۶ چند درصد از هسته لینوکس توسط توروالدز نوشته شد (به عدد)؟"
answer3 = rag_chain.invoke({"query": query})
answer3

In [ ]:
query = "پرسش ۴: چه کسی بنیاد نرم‌افزارهای آزاد را بنا نهاد؟"
answer4 = rag_chain.invoke({"query": query})
answer4

In [ ]:
query = "پرسش ۵: ریچارد استالمن در ۲۱ سالگی در کدام شرکت کار می‌کرد؟"
answer5 = rag_chain.invoke({"query": query})
answer5

In [ ]:
query = "پرسش ۶: یکی از مشهورترین پروژه‌هایی که در ابتدا پروژه‌ی آزاد و آکادمیک بود اما بعد وارد محیط بسته‌ی تجاری شد چه بود؟"
answer6 = rag_chain.invoke({"query": query})
answer6